# LoRA Fine-Tuning: HFACS Extract & Classify (Llama-3.1-8B-Instruct)

This notebook fine-tunes `meta-llama/Llama-3.1-8B-Instruct` with LoRA on the ASRS extract-and-classify task: given a narrative, output the 8 HFACS contributing-factor flags plus `Q1_Error` / `Q2_Violation` / `Final_Class` as JSON.

Training data is generated locally by `src/llm/build_lora_dataset.py` (500 train / 200 test, balanced across Error/Violation/Neither/Both).

## 1. Before you start

1. **Runtime**: `Runtime > Change runtime type > A100 GPU`.
2. **Hugging Face access**: this model is gated. Go to https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, request/accept access, then create a *read* access token at https://huggingface.co/settings/tokens.
3. **Upload data to Google Drive**. From your local repo, upload the two files produced by `src/llm/build_lora_dataset.py`:
   - `data/processed/lora/lora_train.jsonl`
   - `data/processed/lora/lora_test.jsonl`

   into a `data/` subfolder inside your existing Drive folder:
   ```
   MyDrive/llm_hfcas_extract_and_classify/data/lora_train.jsonl
   MyDrive/llm_hfcas_extract_and_classify/data/lora_test.jsonl
   ```
   The `outputs/` folder (adapter weights, predictions) is created automatically during training.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
!pip install -q -U transformers accelerate peft trl datasets huggingface_hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Config ---
DRIVE_DIR = "/content/drive/MyDrive/llm_hfcas_extract_and_classify"
TRAIN_PATH = f"{DRIVE_DIR}/data/lora_train.jsonl"
TEST_PATH = f"{DRIVE_DIR}/data/lora_test.jsonl"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs/llama3.1-8b-extract-classify-lora"

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
MAX_SEQ_LENGTH = 4096
NUM_EPOCHS = 5
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

## 2. Load data

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
test_ds = load_dataset("json", data_files=TEST_PATH, split="train")

print(train_ds)
print(test_ds)
print(train_ds[0]["messages"][-1]["content"])

## 3. Load base model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

## 4. LoRA config

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## 5. Format dataset with the chat template

In [ ]:
def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

train_ds_fmt = train_ds.map(format_example, remove_columns=train_ds.column_names)
print(train_ds_fmt[0]["text"][:1000])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds_fmt,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to: {OUTPUT_DIR}")

## 7. Evaluate on the held-out test set

Generates a prediction for each of the 200 test narratives (system + user messages only), parses the JSON, and compares it to the ground-truth label saved alongside each example.

In [ ]:
import json
from tqdm import tqdm

model.eval()

def generate(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

results = []
for example in tqdm(test_ds):
    prompt_messages = example["messages"][:2]
    gen_text = generate(prompt_messages)
    try:
        pred = json.loads(gen_text)
    except json.JSONDecodeError:
        pred = {}
    true = json.loads(example["messages"][2]["content"])
    results.append({
        "original_index": example["original_index"],
        "raw_output": gen_text,
        "pred": pred,
        "true": true,
    })

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

classes = ["Error", "Violation", "Neither", "Both"]
y_true = [r["true"]["Final_Class"] for r in results]
y_pred = [r["pred"].get("Final_Class", "INVALID") for r in results]

print("Final_Class classification report:")
print(classification_report(y_true, y_pred, labels=classes, zero_division=0))
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred, labels=classes))

invalid = sum(1 for p in y_pred if p not in classes)
print(f"\nInvalid/unparseable outputs: {invalid} / {len(y_pred)}")

In [ ]:
# Per-factor accuracy (the 8 HFACS flags + Q1_Error / Q2_Violation)
factor_keys = [k for k in results[0]["true"] if k != "Final_Class"]
for key in factor_keys:
    correct = sum(1 for r in results if r["pred"].get(key) == r["true"][key])
    print(f"{key}: {correct}/{len(results)} = {correct/len(results):.3f}")

In [ ]:
out_path = f"{OUTPUT_DIR}/test_predictions.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Saved predictions to: {out_path}")